# 01 City Mapping

## Phase 2 Scope

This notebook documents the Phase 2 city mapping and reference model work. It defines the controlled city scope, canonical schema, source mapping rules, validation approach, and handoff to later phases.

Phase 2 is limited to city-reference work. It does not implement EEA ingestion, Wikipedia scraping, Open-Meteo API client behavior, Kafka producer logic, Spark processing, Gold tables, dashboards, or final analysis.

## Phase 2 Deliverables

The Phase 2 city reference deliverables are generated locally only when explicitly called:

- `data/silver/city_reference.csv`
- `data/silver/city_reference.parquet`

Both files are ignored by Git under the repository data policy. They are local reproducible outputs, not committed source data.

## City Reference Scope

The starter scope contains exactly 8 European cities. Vienna and Berlin are carried forward from the Phase 1 pilot checks.

| city_id | city_name | country_code | latitude | longitude | selection_rationale |
| --- | --- | --- | ---: | ---: | --- |
| vienna_at | Vienna | AT | 48.2082 | 16.3738 | Phase 1 pilot city; Open-Meteo, EEA metadata, and Wikipedia feasibility confirmed. |
| berlin_de | Berlin | DE | 52.5200 | 13.4050 | Phase 1 pilot city; Open-Meteo, EEA metadata, and Wikipedia feasibility confirmed. |
| paris_fr | Paris | FR | 48.8566 | 2.3522 | Major European capital with expected monitoring and rich city metadata. |
| madrid_es | Madrid | ES | 40.4168 | -3.7038 | Major European capital; southern European comparison city. |
| rome_it | Rome | IT | 41.9028 | 12.4964 | Major European capital; Mediterranean comparison city. |
| amsterdam_nl | Amsterdam | NL | 52.3676 | 4.9041 | Major European city with expected monitoring and compact urban context. |
| warsaw_pl | Warsaw | PL | 52.2297 | 21.0122 | Major Central/Eastern European capital for regional diversity. |
| prague_cz | Prague | CZ | 50.0755 | 14.4378 | Central European capital with expected source coverage and manageable scope. |

## Canonical Schema

The city reference table is the controlled join surface for later phases. Downstream work must use `city_id`; free-text city names, station names, Wikipedia titles, or display labels are not downstream join keys.

| field | required | purpose |
| --- | --- | --- |
| city_id | yes | Stable technical join key, for example `vienna_at`. |
| city_name | yes | Human-readable display name. |
| city_name_normalized | yes | Lowercase normalized city name used to derive `city_id`. |
| country_code | yes | ISO 3166-1 alpha-2 country code. |
| latitude | yes | Canonical WGS84 city coordinate for Open-Meteo and source alignment. |
| longitude | yes | Canonical WGS84 city coordinate for Open-Meteo and source alignment. |
| population | no | Future contextual metadata, nullable. |
| area_km2 | no | Future contextual metadata, nullable. |
| population_density | no | Future contextual or derived metadata, nullable. |
| mapping_notes | yes | Transparent source-alignment rationale. |
| eea_station_selection_notes | yes | Future EEA station mapping rationale. |
| wikipedia_page_title | no | Wikipedia linkage metadata. |
| wikipedia_url | no | Wikipedia traceability URL. |
| wikipedia_metadata_notes | no | Missing, ambiguous, or reviewed metadata notes. |
| open_meteo_coordinate_notes | no | Coordinate assumptions for future Open-Meteo work. |

## Source Mapping Rules

### EEA Station Mapping

EEA data is station-based, while this project joins sources through `city_id`. Candidate EEA stations must be reviewed by distance to the city reference coordinate, pollutant coverage for PM2.5, PM10, and NO2, time coverage, and station representativeness where available.

The station mapping must be reviewed before Phase 3 ingestion starts. If no station covers all target pollutants, the limitation must be documented and the best reviewed candidate per pollutant should remain `candidate` or `fallback` until approved.

### Wikipedia Metadata Join

Wikipedia metadata is contextual, not official ground truth. Planned contextual fields include `population`, `area_km2`, `population_density`, page title, URL, coordinate comparison notes, country context, and `wikipedia_metadata_notes`.

Missing or ambiguous values stay null with notes. Values must not be guessed from unrelated pages, search snippets, dashboards, or other websites.

### Open-Meteo Coordinate And Field Mapping

Open-Meteo requests are coordinate-based. Later client work must use the city reference `latitude` and `longitude` for each `city_id`.

Field mapping from Phase 1:

| project pollutant | Open-Meteo field |
| --- | --- |
| PM2.5 | `pm2_5` |
| PM10 | `pm10` |
| NO2 | `nitrogen_dioxide` |

Future requests should use `timezone=UTC`, and later events should carry explicit UTC timestamps such as `event_time_utc` and `ingestion_time_utc`.

## Validation Approach

`tests/test_city_mapping.py` protects the Phase 2 city reference model. It checks:

- required columns exist,
- required join-key fields are not null,
- `city_id` values are unique and stable,
- normalized names match `city_id`,
- country codes use two uppercase letters,
- coordinates are valid WGS84 ranges,
- at least 8 starter cities exist,
- generated CSV and Parquet outputs can be read back when the writer is explicitly called.

## Phase 2 Definition Of Done

Phase 2 is ready for a gate review only when:

- `data/silver/city_reference.parquet` is stable as input for later phases,
- downstream joins use `city_id`, not free-text city names,
- EEA, Wikipedia, and Open-Meteo mapping assumptions are documented,
- city reference tests pass,
- no EEA ingestion, Wikipedia parser, Open-Meteo client, Kafka, Spark, Gold table, dashboard, or final analysis work is smuggled into Phase 2.

## Readback Example

The following code is intentionally lightweight and local-only. It shows how the Phase 2 Parquet deliverable can be read once generated.


In [ ]:
from pathlib import Path
import pandas as pd

path = Path('../data/silver/city_reference.parquet')
if path.exists():
    city_reference = pd.read_parquet(path)
    display(city_reference.head())
else:
    print('Generate first with: python -m src.city_mapping.build_city_reference')
